In [7]:
import numpy as np
import scipy.signal as signal
from scipy.linalg import eigh
from scipy.io import wavfile
import IPython.display as ipd
from scipy.signal import resample_poly
import whisper
import soundfile as sf


def isolate_angle_mvdr_like(audio, fs, target_angle_deg, tolerance_deg=25, nperseg=1024):
    """
    Soft directional masking + covariance estimation + MVDR-like beamforming.

    audio: shape (samples, channels)
    returns mono enhanced signal
    """

    eps = 1e-8
    audio = audio.astype(np.float32)

    # STFT: Zxx shape (channels, freqs, frames)
    _, _, Zxx = signal.stft(audio.T, fs=fs, nperseg=nperseg)
    C, F, T = Zxx.shape
    mag = np.abs(Zxx)

    # Same directional energies as your original code
    front_energy = mag[0] + mag[2]
    back_energy  = mag[1] + mag[3]
    left_energy  = mag[0] + mag[1]
    right_energy = mag[2] + mag[3]

    # Direction scores
    fb_score = np.log((front_energy + eps) / (back_energy + eps))
    lr_score = np.log((left_energy + eps) / (right_energy + eps))

    angle_est = np.arctan2(lr_score, fb_score)
    angle_est = np.mod(angle_est, 2 * np.pi)

    target_angle = np.deg2rad(target_angle_deg)
    tolerance = np.deg2rad(tolerance_deg)

    angle_diff = np.angle(np.exp(1j * (angle_est - target_angle)))

    # Soft target mask instead of hard mask
    target_mask = np.exp(-0.5 * (angle_diff / tolerance) ** 2).astype(np.float32)
    noise_mask = 1.0 - target_mask

    enhanced = np.zeros((F, T), dtype=np.complex64)

    ref_mic = 0

    for f in range(F):
        Xf = Zxx[:, f, :]   # shape (C, T)

        # Spatial covariance matrices
        Rs = np.zeros((C, C), dtype=np.complex64)
        Rn = np.zeros((C, C), dtype=np.complex64)

        ms_sum = np.sum(target_mask[f]) + eps
        mn_sum = np.sum(noise_mask[f]) + eps

        for t in range(T):
            x = Xf[:, t:t+1]   # shape (C,1)
            Rs += target_mask[f, t] * (x @ x.conj().T)
            Rn += noise_mask[f, t] * (x @ x.conj().T)

        Rs /= ms_sum
        Rn /= mn_sum

        # Regularization
        Rn += 1e-3 * np.eye(C, dtype=np.complex64)
        Rs += 1e-6 * np.eye(C, dtype=np.complex64)

        # Principal eigenvector of speech covariance as steering vector estimate
        vals, vecs = eigh(Rs)
        d = vecs[:, -1]
        d = d / (d[ref_mic] + eps)

        # MVDR weights
        Rn_inv_d = np.linalg.solve(Rn, d)
        w = Rn_inv_d / (d.conj().T @ Rn_inv_d + eps)

        # Beamform
        enhanced[f, :] = np.conj(w).T @ Xf

    _, y = signal.istft(enhanced, fs=fs)
    y = np.real(y)
    y /= np.max(np.abs(y) + eps)

    return y

fs, data = wavfile.read('Recordings/mixture.wav')
data = data.astype(np.float32) / 32768.0

#angles = np.linspace(0, 360, num=36, endpoint=False)
angles =[7,87,137,184,230,271]



model = whisper.load_model("large")

for angle in angles:
    print(f"Isolating angle {angle}°...")
    enhanced = isolate_angle_mvdr_like(
        data,
        fs,
        target_angle_deg=angle,
        tolerance_deg=25,
        nperseg=1024
    )

    enhanced_16k = resample_poly(enhanced, up=16000, down=fs).astype(np.float32)
    result = model.transcribe(enhanced_16k, fp16=False)
    print(result["text"])

    sf.write(f"speaker_{angle:.0f}deg.wav", enhanced, fs)


#ipd.Audio(enhanced, rate=fs)

Isolating angle 7°...
 Hi, how are you? Great, how are you? I just had my birthday. Really? How old are you? I'm 35. Are you married? Yes, I got married three years ago. Do you have any children? Yes, I have a daughter. She is four years old. Do you have children? Yes, I have two. My older son is five and my younger son is three.
Isolating angle 87°...
 Most sightseers will be Chinese, and for the great majority, this 83 million monument to life down under is the closest they will come to visiting Australia. Australia's Commissioner General to Expo, Lyndall Sash, says, There's a lack of knowledge here about Australia. They just think we blow stuff up out of the earth and export it. Here, pavilions are not as big as they used to be.
Isolating angle 137°...
 Most sightseers will be Chinese, and for the great majority, this 83 million monument to life down under is the closest they will come to visiting Australia. Australia's Commissioner General to Expo, Lyndall Sash, says, There's a lac